## DNS in Kubernetes — CoreDNS and the service name pattern

A ClusterIP is hard to remember; **cluster DNS** makes Services usable. Every cluster ships a DNS server — almost always **CoreDNS** — running as a Deployment in `kube-system` with its own Service. When a pod starts, the kubelet writes `/etc/resolv.conf` pointing at the CoreDNS Service, so every lookup goes there first.

### The Service DNS pattern

```
<service>.<namespace>.svc.cluster.local
```

So `api` in `default` resolves at `api.default.svc.cluster.local`. Because of the search domains in `/etc/resolv.conf`:

- **Same namespace** — the bare name works: `curl http://api`.
- **Across namespaces** — include the namespace: `curl http://api.default`.

### What gets a record

- **Services** → `A`/`AAAA` at the ClusterIP.
- **Headless Services** → `A` records for every Ready pod IP (next section).
- **Named ports** → `SRV` records (`_http._tcp.api.default...`).
- **Pods** → `<ip-with-dashes>.<ns>.pod...`, opt-in only; day to day you use *Service* DNS, not pod DNS.

The `cluster.local` suffix is configurable but universally left alone; the CKA assumes it.

### When DNS is the culprit

Pods reach Services by ClusterIP but `nslookup` fails? Check: are the CoreDNS pods Running/Ready (`-n kube-system -l k8s-app=kube-dns`)? Does the `kube-dns` Service have endpoints? Does a debug pod's `/etc/resolv.conf` point at the kube-dns ClusterIP? Full flow in notebook 10. On our map this is the **CoreDNS** chip in the Networking box — the name layer in front of every Service.